<a href="https://colab.research.google.com/github/sylee15/capstone-mock/blob/main/Cohen's_Kappa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# calculation for validation set

from sklearn.metrics import cohen_kappa_score
import pandas as pd

path_a = '/content/drive/MyDrive/GenAI Final Project Music&Memory/tuning_variant_a_playlists.xlsx'
path_b = '/content/drive/MyDrive/GenAI Final Project Music&Memory/tuning_variant_b_playlists.xlsx'
path_c = '/content/drive/MyDrive/GenAI Final Project Music&Memory/tuning_variant_c_playlists.xlsx'

def compute_kappa(file_path, mabel_tab, sunny_tab, variant_name):
    mabel = pd.read_excel(file_path, sheet_name=mabel_tab)
    sunny = pd.read_excel(file_path, sheet_name=sunny_tab)

    print(f"=== Variant {variant_name} ===")

    # Drop NaN rows
    bio_df = pd.DataFrame({
        'mabel': mabel['biographical_precision_1_5_rater_1'],
        'sunny': sunny['biographical_precision_1_5_rater_2']
    }).dropna()

    db_df = pd.DataFrame({
        'mabel': mabel['database_accuracy_0_1_rater_1'],
        'sunny': sunny['database_accuracy_0_1_rater_2']
    }).dropna()

    kappa_bio = cohen_kappa_score(
        bio_df['mabel'].astype(int),
        bio_df['sunny'].astype(int),
        weights='quadratic'
    )

    kappa_db = cohen_kappa_score(
        db_df['mabel'].astype(int),
        db_df['sunny'].astype(int)
    )

    print(f"\nBiographical Precision Kappa (quadratic weighted): {kappa_bio:.3f}")
    print(f"Database Accuracy Kappa (unweighted):              {kappa_db:.3f}")
    print()
    return kappa_bio, kappa_db

compute_kappa(path_a, 'MABEL_tuning_variant_a_playlist', 'SUNNY_tuning_variant_a_playlist', 'A')
compute_kappa(path_b, 'MABEL_tuning_variant_b_playlist', 'SUNNY_tuning_variant_b_playlist', 'B')
compute_kappa(path_c, 'MABEL_tuning_variant_c_playlist', 'SUNNY_tuning_variant_c_playlist', 'C')

print("Database accuracy reading NaN means 100% (both raters, all profiles). Kappa not applicable due to perfect agreement.")
print()

=== Variant A ===

Biographical Precision Kappa (quadratic weighted): 0.592
Database Accuracy Kappa (unweighted):              1.000

=== Variant B ===

Biographical Precision Kappa (quadratic weighted): 0.571
Database Accuracy Kappa (unweighted):              nan



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


=== Variant C ===

Biographical Precision Kappa (quadratic weighted): 0.773
Database Accuracy Kappa (unweighted):              nan

Database accuracy reading NaN means 100% (both raters, all profiles). Kappa not applicable due to perfect agreement.



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


In [4]:
# calculation from test set

from sklearn.metrics import cohen_kappa_score
import pandas as pd

path = '/content/drive/MyDrive/GenAI Final Project Music&Memory/Human Eval Phase 9.xlsx'

mabel = pd.read_excel(path, sheet_name='Mabel')
sunny = pd.read_excel(path, sheet_name='Sunny')

print("Columns:", mabel.columns.tolist())
print(f"Total rows - Mabel: {len(mabel)}, Sunny: {len(sunny)}")

for variant in ['A', 'B', 'C']:
    print(f"\n=== Variant {variant} ===")

    m = mabel[mabel['variant'] == variant].copy()
    s = sunny[sunny['variant'] == variant].copy()

    # Biographical precision — drop rows where either rater left blank
    bio_df = pd.DataFrame({
        'mabel': m['biographical_precision_1_5_rater_1'].values,
        'sunny': s['biographical_precision_1_5_rater_2'].values
    }).dropna()

    # Database accuracy — drop rows where either rater left blank
    db_df = pd.DataFrame({
        'mabel': m['database_accuracy_0_1_rater_1'].values,
        'sunny': s['database_accuracy_0_1_rater_2'].values
    }).dropna()

    print(f"Bio precision rows: {len(bio_df)}, DB accuracy rows: {len(db_df)}")

    # Biographical precision — quadratic weighted kappa (1-5 scale)
    if len(bio_df) > 0 and bio_df['mabel'].nunique() > 1 and bio_df['sunny'].nunique() > 1:
        kappa_bio = cohen_kappa_score(
            bio_df['mabel'].astype(int),
            bio_df['sunny'].astype(int),
            weights='quadratic'
        )
        print(f"Biographical Precision Kappa (quadratic weighted): {kappa_bio:.3f}")
    else:
        print("Biographical Precision Kappa: N/A (no variance in ratings)")

    # Database accuracy — unweighted kappa (0/1)
    if len(db_df) > 0 and db_df['mabel'].nunique() > 1 and db_df['sunny'].nunique() > 1:
        kappa_db = cohen_kappa_score(
            db_df['mabel'].astype(int),
            db_df['sunny'].astype(int)
        )
        print(f"Database Accuracy Kappa (unweighted):              {kappa_db:.3f}")
    else:
        print("Database Accuracy Kappa: N/A (no variance — all 1s)")

    # Mean scores per rater
    print(f"Avg Bio Precision  — Mabel: {bio_df['mabel'].mean():.2f}, Sunny: {bio_df['sunny'].mean():.2f}")
    print(f"Avg DB Accuracy    — Mabel: {db_df['mabel'].mean():.2f}, Sunny: {db_df['sunny'].mean():.2f}")

Columns: ['profile_id', 'patient_name', 'gender', 'birth_year', 'cultural_background', 'hometown', 'variant', 'rank', 'song', 'artist', 'year', 'relevance_reason', 'biographical_precision_1_5_rater_1', 'database_accuracy_0_1_rater_1', 'notes']
Total rows - Mabel: 900, Sunny: 900

=== Variant A ===
Bio precision rows: 30, DB accuracy rows: 100
Biographical Precision Kappa (quadratic weighted): 0.232
Database Accuracy Kappa (unweighted):              1.000
Avg Bio Precision  — Mabel: 4.30, Sunny: 3.47
Avg DB Accuracy    — Mabel: 0.96, Sunny: 0.96

=== Variant B ===
Bio precision rows: 30, DB accuracy rows: 100
Biographical Precision Kappa (quadratic weighted): 0.569
Database Accuracy Kappa: N/A (no variance — likely all 1s)
Avg Bio Precision  — Mabel: 3.43, Sunny: 3.50
Avg DB Accuracy    — Mabel: 1.00, Sunny: 1.00

=== Variant C ===
Bio precision rows: 30, DB accuracy rows: 100
Biographical Precision Kappa (quadratic weighted): 0.687
Database Accuracy Kappa: N/A (no variance — likely all